In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional,GlobalAveragePooling1D, Activation
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import tensorflow as tf
import cv2
import numpy as np
import os
import glob
import json
from tensorflow import keras
from sklearn.model_selection import train_test_split

In [2]:
# 1. Các biến cấu hình
DATA_PATH        = 'Data'
LABEL_MAP_PATH   = 'Logs/label_map.json'
BATCH_SIZE       = 32
AUTOTUNE         = tf.data.AUTOTUNE

# 2. Load label_map & đếm số lượng nhãn tự động
with open(LABEL_MAP_PATH, 'r', encoding='utf-8') as f:
    label_map = json.load(f)
NUM_CLASSES = len(label_map)
print(f"Tổng số nhãn dự đoán: {NUM_CLASSES}")

# 3. Load danh sách file CHỈ CHO TẬP TRAIN
train_files = glob.glob(os.path.join(DATA_PATH, 'Train', '**', '*.npz'), recursive=True)
print(f"Train samples (Gốc + Augment): {len(train_files)}")

# 4. Hàm parse mỗi file .npz
def _load_npz(path):
    npz_path = path.decode('utf-8')
    data = np.load(npz_path)
    seq   = data['sequence'].astype(np.float32)
    lbl   = np.int32(data['label'])
    return seq, lbl

def parse_fn(path):
    seq, lbl = tf.numpy_function(
        func=_load_npz,
        inp=[path],
        Tout=[tf.float32, tf.int32]
    )
    # Lưu ý: Cập nhật lại shape 201 cho đúng với số keypoint của bạn
    seq.set_shape([60, 201])
    lbl.set_shape([])
    return seq, lbl

def make_dataset(file_list, shuffle=False, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices(file_list)
    if shuffle:
        ds = ds.shuffle(len(file_list), reshuffle_each_iteration=True)
    if repeat:
        ds = ds.repeat()
    ds = ds.map(parse_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds

# 5. Tạo Dataset (Chỉ Train)
train_ds = make_dataset(train_files, shuffle=True, repeat=True)

# 6. Compute steps
steps_per_epoch = len(train_files) // BATCH_SIZE

Tổng số nhãn dự đoán: 100
Train samples (Gốc + Augment): 5100


In [3]:
inputs = tf.keras.Input(shape=(60, 201))

# Khối LSTM thứ nhất
x = Bidirectional(LSTM(256, return_sequences=True, dropout=0.3))(inputs)
x = BatchNormalization()(x)

# Khối LSTM thứ hai
x = Bidirectional(LSTM(256, return_sequences=True, dropout=0.3))(x)
x = BatchNormalization()(x)

# Khối LSTM thứ ba
x = Bidirectional(LSTM(256, dropout=0.3))(x)
x = BatchNormalization()(x)

# Các lớp Dense
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)

# Lớp đầu ra (Dùng NUM_CLASSES để mô hình tự động nhận diện số từ vựng hiện có)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs)
# Biên dịch mô hình
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [4]:
# 1. Tạo thư mục lưu checkpoint (nếu chưa có)
checkpoint_dir = 'Models/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'final_model.keras')

# 2. Khởi tạo callbacks
callbacks = [
    # Lưu mô hình với loss thấp nhất
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='loss',               # ĐÃ SỬA: Đổi từ val_loss sang loss
        save_best_only=True,
        save_weights_only=False,      # lưu cả kiến trúc + weights
        verbose=1
    ),
    # Dừng training nếu 10 epoch liên tiếp không cải thiện loss
    EarlyStopping(
        monitor='loss',               # ĐÃ SỬA: Đổi từ val_loss sang loss
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

In [ ]:
model.fit(
    train_ds,
    epochs=100,
    steps_per_epoch=steps_per_epoch,
    callbacks = callbacks
    # ĐÃ XÓA: validation_data và validation_steps
)

Epoch 1/100
 16/159 [==>...........................] - ETA: 6:23 - loss: 5.1622 - accuracy: 0.0117